In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
import joblib

In [2]:
X = np.load('models/X_class.npy')
y = np.load('models/y_target.npy')

In [3]:
# Handle small demo dataset indexing safely for training split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

print("Training Random Forest Matching Classifier...")
rf_matcher = RandomForestClassifier(n_estimators=50, random_state=42)
rf_matcher.fit(X_train, y_train)

preds = rf_matcher.predict(X_test)
print(f"Baseline Accuracy: {accuracy_score(y_test, preds)}")

# Save the matching model
joblib.dump(rf_matcher, 'models/resume_matcher_rf.pkl')

Training Random Forest Matching Classifier...
Baseline Accuracy: 0.5677216434047572


['models/resume_matcher_rf.pkl']

In [4]:
df = pd.read_csv('data/processed_resume_match.csv')
y = df['result'].values

max_words = 10000
max_len = 400

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['input_text'].fillna(""))
joblib.dump(tokenizer, 'models/nn_tokenizer.pkl')

X_seq = tokenizer.texts_to_sequences(df['input_text'].fillna(""))
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.3, random_state=42)

In [5]:
# Bidirectional LSTM to evaluate relations between JD keywords and Resume highlights
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Binary output: Probability of selection
])

d:\Udemy\resume screener\venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [6]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [7]:
model.fit(X_train, y_train, epochs=5, batch_size=2, validation_data=(X_test, y_test))

Epoch 1/5
3561/3561 ━━━━━━━━━━━━━━━━━━━━ 938s 261ms/step - accuracy: 0.5277 - loss: 0.6902 - val_accuracy: 0.5382 - val_loss: 0.6877
Epoch 2/5
3561/3561 ━━━━━━━━━━━━━━━━━━━━ 852s 239ms/step - accuracy: 0.5883 - loss: 0.6417 - val_accuracy: 0.5480 - val_loss: 0.6450
Epoch 3/5
3561/3561 ━━━━━━━━━━━━━━━━━━━━ 790s 222ms/step - accuracy: 0.6343 - loss: 0.5930 - val_accuracy: 0.5490 - val_loss: 0.6883
Epoch 4/5
3561/3561 ━━━━━━━━━━━━━━━━━━━━ 825s 232ms/step - accuracy: 0.6794 - loss: 0.5434 - val_accuracy: 0.5457 - val_loss: 0.7097
Epoch 5/5
3561/3561 ━━━━━━━━━━━━━━━━━━━━ 867s 244ms/step - accuracy: 0.7297 - loss: 0.4831 - val_accuracy: 0.5500 - val_loss: 0.8225


In [8]:
model.save('models/resume_matcher_lstm.h5')
print("Sequence model saved.")

Sequence model saved.
